In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("anthropic:claude-sonnet-4-5-20250929", temperature=0)
response = llm.invoke("Hola, como estas ?")
response.pretty_print()

================================== Ai Message ==================================

¡Hola! Estoy bien, gracias por preguntar. 😊 

¿Cómo estás tú? ¿En qué puedo ayudarte hoy?


In [11]:
response = llm.invoke("Que clima hace en la ciudad de Bogota ?")
response.pretty_print()

================================== Ai Message ==================================

# Clima en Bogotá

Bogotá tiene un **clima de montaña tropical** con las siguientes características generales:

## Temperatura
- **Promedio anual**: 14°C (57°F)
- **Máximas**: 19-20°C durante el día
- **Mínimas**: 7-9°C durante la noche

## Características principales
- ☁️ **Nublado frecuentemente**
- 🌧️ **Lluvias regulares** durante todo el año
- 🌡️ **Clima fresco** debido a su altitud (2,640 metros sobre el nivel del mar)
- 🌤️ **Poca variación** de temperatura entre estaciones

## Temporadas
- **Temporada seca**: Diciembre-Marzo y Julio-Agosto
- **Temporada lluviosa**: Abril-Mayo y Septiembre-Noviembre

## Recomendación
Se aconseja llevar siempre **ropa abrigada y paraguas**, ya que el clima puede cambiar varias veces en un mismo día.

---
*Nota: Para conocer el clima actual en tiempo real, te recomendaría consultar un servicio meteorológico actualizado.*


In [12]:
SYSTEM_PROMPT = """
Eres un asistente en nuestra tiendo y los productos que vendemos son los siguientes:
- Camisetas
- Pantalones
- Medias
- Zapatillas
"""

messages = [
    ("system", SYSTEM_PROMPT),
    ("user", "Que productos tienen disponibles en la tienda ?")
]

response = llm.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================

¡Hola! Bienvenido a nuestra tienda. 😊

Actualmente tenemos disponibles los siguientes productos:

1. **Camisetas**
2. **Pantalones**
3. **Medias**
4. **Zapatillas**

¿Te interesa alguno de estos productos en particular? Estaré encantado de ayudarte con más información o cualquier consulta que tengas.


In [13]:
from langchain_core.tools import tool
import requests

@tool("get_products", description="Obtiene la lista de productos disponibles en la tienda.")
def get_products(price: int):
    """Obtiene la lista de productos disponibles en la tienda."""
    products = [
        {"nombre": "Camisetas", "precio": 20},
        {"nombre": "Pantalones", "precio": 30},
        {"nombre": "Medias", "precio": 10},
        {"nombre": "Zapatillas", "precio": 50}
    ]
    return "".join([f"{product['nombre']}: ${product['precio']} \n" for product in products])

In [14]:
print(get_products.invoke({"price": 50}))

Camisetas: $20 
Pantalones: $30 
Medias: $10 
Zapatillas: $50 



In [15]:
from langchain_core.tools import tool
import requests

@tool("get_products", description="Obtiene la lista de productos disponibles en la tienda.")
def get_products():
    #Connect with API to get the products
    """Obtiene la lista de productos disponibles en la tienda."""
    response = requests.get("https://api.escuelajs.co/api/v1/products")
    products = response.json()
    return "".join([f"{product['title']}: ${product['price']} \n" for product in products])

In [16]:
print(get_products.invoke({}))

Majestic Mountain Graphic T-Shirt: $44 
Classic Red Pullover Hoodie: $10 
Classic Heather Gray Hoodie: $69 
Classic Grey Hooded Sweatshirt: $90 
Classic Black Hooded Sweatshirt: $79 
Classic Comfort Fit Joggers: $25 
Classic Comfort Drawstring Joggers: $79 
Classic Red Jogger Sweatpants: $98 
Classic Navy Blue Baseball Cap: $61 
Classic Blue Baseball Cap: $86 
Classic Red Baseball Cap: $35 
Classic Black Baseball Cap: $58 
Classic Olive Chino Shorts: $84 
Classic High-Waisted Athletic Shorts: $43 
Classic White Crew Neck T-Shirt: $39 
Classic White Tee - Timeless Style and Comfort: $73 
Classic Black T-Shirt: $35 
Sleek White & Orange Wireless Gaming Controller: $69 
Sleek Wireless Headphone & Inked Earbud Set: $44 
Sleek Comfort-Fit Over-Ear Headphones: $28 
Efficient 2-Slice Toaster: $48 
Sleek Wireless Computer Mouse: $10 
Sleek Modern Laptop with Ambient Lighting: $43 
Sleek Modern Laptop for Professionals: $97 
Stylish Red & Silver Over-Ear Headphones: $39 
Sleek Mirror Finish Pho

In [17]:
from langchain_core.tools import tool
import requests

@tool("get_weather", description="Get a Weather frof a City")
def get_weather(city: str):
    response    = requests.get(f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1")
    data      = response.json()
    latitude = data['results'][0]['latitude']
    longitude = data['results'][0]['longitude']

    response  = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true")
    data    = response.json()

    return f"The weather on the city {city.title()} is {data['current_weather']['temperature']}"

In [ ]:
#print(result.get("results")[0]['latitude'])

In [18]:
result = get_weather.invoke({"city":"bogota"})
#result["current_weather"]["temperature"]
#result['results'][0]['latitude']
#result['results'][0]['latitude']
#result['results'][0]['longitude']
result

'The weather on the city Bogota is 11.6'

In [29]:
SYSTEM_PROMPT = """
Eres un asistente de ventas que ayuda a los clientes a encontrar los productos
en nuestra tiendo y dar el clima, los productos que vendemos son los siguientes:

- Camisetas
- Pantalones
- Medias
- Zapatillas

Tus tools son:
- get_products: para obtener los productos disponibles en la tienda.
- get_weather: para obtener el clima de la ciudad. Siempre debes pasar como parametro
la ciudad en minuscula y sin acentos.

"""

messages = [
    ("system", SYSTEM_PROMPT),
    ("user", "Que productos tienen disponibles en la tienda ?")
]

llm_with_tools = llm.bind_tools([get_products, get_weather])

response = llm_with_tools.invoke(messages)
print(response.pretty_print())
print(response.tool_calls)

================================== Ai Message ==================================

[{'id': 'toolu_01SCdPaFQugDQ2Uqyv99nZAJ', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'get_products', 'type': 'tool_use', 'toolset_name': None}]
Tool Calls:
  get_products (toolu_01SCdPaFQugDQ2Uqyv99nZAJ)
 Call ID: toolu_01SCdPaFQugDQ2Uqyv99nZAJ
  Args:
None
[{'name': 'get_products', 'args': {}, 'id': 'toolu_01SCdPaFQugDQ2Uqyv99nZAJ', 'type': 'tool_call'}]


In [27]:
messages = [
    ("system", SYSTEM_PROMPT),
    ("user", "Hola, que tal ?")
]

response = llm_with_tools.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================

¡Hola! ¡Qué tal! 😊 

Bienvenido/a a nuestra tienda. Estoy aquí para ayudarte a encontrar los productos que necesites o darte información sobre el clima si lo deseas.

Vendemos:
- **Camisetas**
- **Pantalones**
- **Medias**
- **Zapatillas**

¿En qué puedo ayudarte hoy? ¿Buscas algún producto en particular o quieres saber qué tenemos disponible?


In [30]:
messages = [
    ("system", SYSTEM_PROMPT),
    ("user", "Cual es el clima de Bogota?")
]

response = llm_with_tools.invoke(messages)
print(response.pretty_print())
print(response.tool_calls)

================================== Ai Message ==================================

[{'id': 'toolu_01Xqsj3Kjvkfs8fhaTCDWbvR', 'caller': {'type': 'direct'}, 'input': {'city': 'bogota'}, 'name': 'get_weather', 'type': 'tool_use', 'toolset_name': None}]
Tool Calls:
  get_weather (toolu_01Xqsj3Kjvkfs8fhaTCDWbvR)
 Call ID: toolu_01Xqsj3Kjvkfs8fhaTCDWbvR
  Args:
    city: bogota
None
[{'name': 'get_weather', 'args': {'city': 'bogota'}, 'id': 'toolu_01Xqsj3Kjvkfs8fhaTCDWbvR', 'type': 'tool_call'}]
